In [1]:
!pip install axelrod

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 39.7 MB/s eta 0:00:00


In [2]:
"""
IPD Strategy — Enhanced Dueling Double DQN  (publishable quality + verbose)
============================================================================

Key upgrades over original fast version:
  QUALITY
  -------
  1. Reward shaping: normalised per-round reward + cooperative bonus signal.
  2. Larger network (256 hidden, 3-layer shared trunk with GELU).
  3. PER alpha=0.7, beta annealed over 60k steps.
  4. AdamW + weight decay for better generalisation.
  5. Longer training (20k episodes) with higher warmup (4k).
  6. Ensemble eval: 150 eps for stochastic/evolved opponents.
  7. Cosine LR with warm restarts.

  BUG FIXES vs original
  ---------------------
  * _seed_player: was setting p._random = np.random.RandomState(...) which
    breaks axelrod >= 4.x. Fixed to use axelrod's own RandomGenerator via
    p.set_seed() so random_choice() works correctly on all stochastic players.
  * MetaWinner removed: its sub-player Appeaser reads opponent.history[-1]
    on round 0, causing IndexError in step-by-step play. It is fundamentally
    incompatible with the single-step IPDEnv architecture.
  * All 33 remaining opponents individually smoke-tested before inclusion.

  OBSERVABILITY
  -------------
  * tqdm bar: outer = episodes, live postfix AvgR / Loss / beta / LR / buf.
  * Snapshot every VERBOSE_EVERY episodes: quick eval + per-family breakdown.
  * Final ablation-style table: DQN vs every baseline, per-family subtotals,
    overall mean, delta column vs DQN.
  * JSON saved to ./results/ipd_dqn_v2.json
"""

import os, random, time, json, warnings, collections, math
from typing import Dict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

warnings.filterwarnings("ignore")
import axelrod as axl

# ── constants ─────────────────────────────────────────────────────────────────
SEED          = 42
C, D          = 0, 1
ROUNDS        = 100
PAYOFF        = {(C, C): 3, (C, D): 0, (D, C): 5, (D, D): 1}
FEAT_DIM      = 32
N_STEP        = 4
GAMMA         = 0.97
HIDDEN        = 256
TOTAL_EPS     = 20_000
WARMUP_EPS    = 4_000
BATCH         = 512
BUF_CAP       = 150_000
TGT_UPD       = 1_000
LR            = 3e-4
BETA_STEPS    = 60_000
VERBOSE_EVERY = 2_000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seeds(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seeds()

def a2i(a): return 0 if a == axl.Action.C else 1
def i2a(i): return axl.Action.C if i == 0 else axl.Action.D
def _log(msg): tqdm.write(f"  [IPD] {msg}")


# ── Reward shaping ─────────────────────────────────────────────────────────────
def shape_reward(raw: float, my_a: int, opp_a: int) -> float:
    norm       = (raw - 2.5) / 2.5
    coop_bonus = 0.1 if (my_a == C and opp_a == C) else 0.0
    return float(np.clip(norm + coop_bonus, -1.5, 1.5))


# ── Opponent seeding ───────────────────────────────────────────────────────────
_GAME        = axl.Game()
_MATCH_ATTRS = {"length": ROUNDS, "game": _GAME, "noise": 0}

def _seed_player(p):
    """
    Correctly seed an axelrod player using axelrod's own RandomGenerator.

    axelrod strategies call self._random.random_choice() internally.
    self._random must be a axelrod.random_.RandomGenerator instance.
    The original code used np.random.RandomState which lacks random_choice()
    and causes AttributeError on any stochastic opponent.

    Fix: call p.set_seed(int) which does:
        self._random = RandomGenerator(seed=self._seed)
    """
    if hasattr(p, "set_seed"):
        p.set_seed(random.randint(0, 2**31))
    if hasattr(p, "set_match_attributes"):
        p.set_match_attributes(**_MATCH_ATTRS)
    # EvolvedHMM5 contains internal SimpleHMM sub-objects
    for attr in ("hmm", "hmm_C", "hmm_D"):
        sub = getattr(p, attr, None)
        if sub is not None and hasattr(sub, "set_seed"):
            sub.set_seed(random.randint(0, 2**31))
    # AdaptiveTitForTat and similar have a 'team' list
    for sub in getattr(p, "team", None) or []:
        if hasattr(sub, "set_seed"):
            sub.set_seed(random.randint(0, 2**31))
        if hasattr(sub, "set_match_attributes"):
            sub.set_match_attributes(**_MATCH_ATTRS)


# ── Incremental Feature State (O(1) per step) ─────────────────────────────────
class FeatureState:
    __slots__ = [
        "ph", "oh", "rnd", "opp_d_sum", "my_d_sum",
        "early_d_sum", "early_n", "late_d_sum", "late_n",
        "oh5", "oh5_sum", "oh10", "oh10_sum",
        "ovar_sum", "ovar_sq_sum",
        "after_myC_oppC", "after_myC_oppD", "after_myC_n",
        "after_myD_oppC", "after_myD_oppD", "after_myD_n",
        "mirror_sum", "mirror_n", "ever_defected", "all_D_after_first",
        "alt_sum", "alt_n", "mc_sum", "ex_sum", "md_sum",
        "mc5_buf", "mc5_sum", "mc_streak", "d_streak",
        "forg_sum", "forg_n", "cyc2_match", "cyc2_n", "cyc3_match", "cyc3_n",
        "r0", "r1", "r2", "opp_last",
    ]

    def __init__(self):
        self.ph = []; self.oh = []; self.rnd = 0
        self.opp_d_sum = 0; self.my_d_sum = 0
        self.early_d_sum = 0; self.early_n = 0
        self.late_d_sum  = 0; self.late_n  = 0
        self.oh5  = collections.deque(maxlen=5);  self.oh5_sum  = 0
        self.oh10 = collections.deque(maxlen=10); self.oh10_sum = 0
        self.ovar_sum = 0.0; self.ovar_sq_sum = 0.0
        self.after_myC_oppC = 0; self.after_myC_oppD = 0; self.after_myC_n = 0
        self.after_myD_oppC = 0; self.after_myD_oppD = 0; self.after_myD_n = 0
        self.mirror_sum = 0; self.mirror_n = 0
        self.ever_defected = False; self.all_D_after_first = True
        self.alt_sum = 0; self.alt_n = 0
        self.mc_sum = 0; self.ex_sum = 0; self.md_sum = 0
        self.mc5_buf = collections.deque(maxlen=5); self.mc5_sum = 0
        self.mc_streak = 0; self.d_streak = 0
        self.forg_sum = 0; self.forg_n = 0
        self.cyc2_match = 0; self.cyc2_n = 0
        self.cyc3_match = 0; self.cyc3_n = 0
        self.r0 = 0.5; self.r1 = 0.5; self.r2 = 0.5; self.opp_last = 0.5

    def update(self, my_a: int, opp_a: int):
        t        = self.rnd
        prev_my  = self.ph[-1] if self.ph else None
        prev_opp = self.oh[-1] if self.oh else None

        if   t == 0: self.r0 = float(opp_a == D)
        elif t == 1: self.r1 = float(opp_a == D)
        elif t == 2: self.r2 = float(opp_a == D)
        self.opp_last = float(opp_a)

        if len(self.oh5)  == 5:  self.oh5_sum  -= self.oh5[0]
        if len(self.oh10) == 10: self.oh10_sum -= self.oh10[0]
        self.oh5.append(opp_a);  self.oh5_sum  += opp_a
        self.oh10.append(opp_a); self.oh10_sum += opp_a

        if t < 20: self.early_d_sum += opp_a; self.early_n += 1
        else:      self.late_d_sum  += opp_a; self.late_n  += 1

        if opp_a == D: self.d_streak += 1
        else:          self.d_streak  = 0

        v = float(1 - opp_a); self.ovar_sum += v; self.ovar_sq_sum += v * v

        if prev_my is not None:
            if prev_my == C:
                self.after_myC_n += 1
                if opp_a == C: self.after_myC_oppC += 1
                else:          self.after_myC_oppD += 1
            else:
                self.after_myD_n += 1
                if opp_a == C: self.after_myD_oppC += 1
                else:          self.after_myD_oppD += 1

        if prev_opp is not None:
            self.mirror_n += 1
            if opp_a == prev_my: self.mirror_sum += 1
            self.alt_n += 1
            if opp_a != prev_opp: self.alt_sum += 1

        if opp_a == D and not self.ever_defected: self.ever_defected = True
        if self.ever_defected and opp_a == C:     self.all_D_after_first = False

        mc_now = int(my_a == C and opp_a == C)
        if len(self.mc5_buf) == 5: self.mc5_sum -= self.mc5_buf[0]
        self.mc5_buf.append(mc_now); self.mc5_sum += mc_now
        if mc_now: self.mc_sum += 1; self.mc_streak += 1
        else:      self.mc_streak = 0
        if my_a == D and opp_a == C: self.ex_sum += 1
        if my_a == D and opp_a == D: self.md_sum += 1

        if prev_my is not None and prev_my == D and prev_opp == D:
            self.forg_n += 1
            if opp_a == C: self.forg_sum += 1

        if t >= 2:
            self.cyc2_n += 1
            if opp_a == self.oh[t - 2]: self.cyc2_match += 1
        if t >= 3:
            self.cyc3_n += 1
            if opp_a == self.oh[t - 3]: self.cyc3_match += 1

        self.opp_d_sum += opp_a; self.my_d_sum += my_a
        self.ph.append(my_a); self.oh.append(opp_a); self.rnd += 1

    def features(self) -> np.ndarray:
        t  = self.rnd; n = max(1, t); ph = self.ph
        opp_dr = self.opp_d_sum / n; my_dr = self.my_d_sum / n
        cac = (self.after_myC_oppC / self.after_myC_n) if self.after_myC_n else 0.5
        dac = (self.after_myC_oppD / self.after_myC_n) if self.after_myC_n else 0.5
        cad = (self.after_myD_oppC / self.after_myD_n) if self.after_myD_n else 0.5
        dad = (self.after_myD_oppD / self.after_myD_n) if self.after_myD_n else 0.5
        mir  = (self.mirror_sum / self.mirror_n) if self.mirror_n else 0.5
        pret = float(self.ever_defected and self.all_D_after_first)
        alt  = (self.alt_sum / self.alt_n) if self.alt_n else 0.0
        if n >= 3:
            mv = self.ovar_sum / n
            ovar = max(0.0, self.ovar_sq_sum / n - mv * mv)
        else:
            ovar = 0.25
        last5_dr  = (self.oh5_sum  / len(self.oh5))  if len(self.oh5)  >= 5  else opp_dr
        last10_dr = (self.oh10_sum / len(self.oh10)) if len(self.oh10) >= 10 else opp_dr
        early = (self.early_d_sum / self.early_n) if self.early_n else 0.5
        late  = (self.late_d_sum  / self.late_n)  if self.late_n  else (opp_dr if t > 0 else 0.5)
        mc  = self.mc_sum / n; ex = self.ex_sum / n; md = self.md_sum / n
        cyc2 = (self.cyc2_match / self.cyc2_n) if self.cyc2_n else 0.5
        cyc3 = (self.cyc3_match / self.cyc3_n) if self.cyc3_n else 0.5
        forg = (self.forg_sum / self.forg_n)   if self.forg_n else 0.5
        mc5  = (self.mc5_sum  / len(self.mc5_buf)) if len(self.mc5_buf) >= 5 else mc
        my_last3 = [float(ph[-i-1] == D) if len(ph) > i else 0.5 for i in range(3)]
        return np.array([
            opp_dr, my_dr, self.r0, self.r1, self.r2,
            cac, dac, cad, dad, mir, pret, alt, ovar,
            last5_dr, last10_dr, min(self.d_streak / 10., 1.),
            early, late, mc, ex, md, cyc2, cyc3, forg, mc5,
            my_last3[0], my_last3[1], my_last3[2],
            t / ROUNDS, min(t / 20., 1.),
            self.opp_last, min(self.mc_streak / 10., 1.),
        ], dtype=np.float32)

def extract_features_init() -> np.ndarray:
    return np.full(FEAT_DIM, 0.5, dtype=np.float32)


# ── NoisyLinear ────────────────────────────────────────────────────────────────
class NoisyLinear(nn.Module):
    def __init__(self, in_f: int, out_f: int, sigma0: float = 0.4):
        super().__init__()
        self.in_f = in_f; self.out_f = out_f
        self.w_mu  = nn.Parameter(torch.empty(out_f, in_f))
        self.w_sig = nn.Parameter(torch.empty(out_f, in_f))
        self.b_mu  = nn.Parameter(torch.empty(out_f))
        self.b_sig = nn.Parameter(torch.empty(out_f))
        self.register_buffer("w_eps", torch.empty(out_f, in_f))
        self.register_buffer("b_eps", torch.empty(out_f))
        self.sigma0 = sigma0
        self.reset_parameters(); self.sample_noise()

    def reset_parameters(self):
        b = 1 / math.sqrt(self.in_f)
        self.w_mu.data.uniform_(-b, b); self.b_mu.data.uniform_(-b, b)
        self.w_sig.data.fill_(self.sigma0 / math.sqrt(self.in_f))
        self.b_sig.data.fill_(self.sigma0 / math.sqrt(self.out_f))

    @staticmethod
    def _f(x): return x.sign() * x.abs().sqrt()

    def sample_noise(self):
        ei = self._f(torch.randn(self.in_f,  device=self.w_eps.device))
        eo = self._f(torch.randn(self.out_f, device=self.b_eps.device))
        self.w_eps.copy_(eo.outer(ei)); self.b_eps.copy_(eo)

    def forward(self, x):
        if self.training:
            return F.linear(x,
                            self.w_mu + self.w_sig * self.w_eps,
                            self.b_mu + self.b_sig * self.b_eps)
        return F.linear(x, self.w_mu, self.b_mu)


# ── Deeper Dueling DQN (3-layer trunk, GELU) ──────────────────────────────────
class DuelingDQN(nn.Module):
    def __init__(self, in_dim: int = FEAT_DIM, n_actions: int = 2, hidden: int = HIDDEN):
        super().__init__()
        h2 = hidden // 2
        self.shared = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, h2),     nn.LayerNorm(h2),     nn.GELU(),
        )
        self.val = nn.Sequential(NoisyLinear(h2, h2), nn.GELU(), NoisyLinear(h2, 1))
        self.adv = nn.Sequential(NoisyLinear(h2, h2), nn.GELU(), NoisyLinear(h2, n_actions))

    def forward(self, x):
        h = self.shared(x); v = self.val(h); a = self.adv(h)
        return v + a - a.mean(dim=-1, keepdim=True)

    def sample_noise(self):
        for m in self.modules():
            if isinstance(m, NoisyLinear): m.sample_noise()


# ── Prioritised Experience Replay ──────────────────────────────────────────────
Tr = collections.namedtuple("Tr", ["s", "a", "r", "s2", "done"])

class PERBuffer:
    def __init__(self, cap: int = BUF_CAP, alpha: float = 0.7):
        self.cap = cap; self.alpha = alpha
        self.buf   = [None] * cap
        self.prios = np.zeros(cap, dtype=np.float32)
        self.pos = 0; self.size = 0

    def push(self, *args, prio: float = 1.0):
        self.buf[self.pos]   = Tr(*args)
        self.prios[self.pos] = prio ** self.alpha
        self.pos  = (self.pos + 1) % self.cap
        self.size = min(self.size + 1, self.cap)

    def sample(self, n: int, beta: float = 0.4):
        p    = self.prios[:self.size]
        p    = p / p.sum()
        idxs = np.random.choice(self.size, n, replace=False, p=p)
        w    = (self.size * p[idxs]) ** (-beta)
        w   /= w.max()
        return [self.buf[i] for i in idxs], idxs, w.astype(np.float32)

    def update_prios(self, idxs, new_prios):
        self.prios[idxs] = (np.abs(new_prios) + 1e-6) ** self.alpha

    def __len__(self): return self.size


# ── IPDEnv ─────────────────────────────────────────────────────────────────────
class IPDEnv:
    def __init__(self, opp_class, opp_kwargs: dict = None):
        self.opp_class  = opp_class
        self.opp_kwargs = opp_kwargs or {}
        self.reset()

    def reset(self) -> np.ndarray:
        self.opp = self.opp_class(**self.opp_kwargs)
        self.opp.reset()
        _seed_player(self.opp)
        self.agent_proxy = axl.Cooperator()
        self.agent_proxy.reset()
        self.fs = FeatureState()
        return extract_features_init()

    def step(self, action: int):
        opp_action = a2i(self.opp.strategy(self.agent_proxy))
        raw        = float(PAYOFF[(action, opp_action)])
        rew        = shape_reward(raw, action, opp_action)

        self.agent_proxy.history.append(i2a(action),     i2a(opp_action))
        self.opp.history.append(        i2a(opp_action), i2a(action))

        self.fs.update(action, opp_action)
        obs  = self.fs.features()
        done = self.fs.rnd >= ROUNDS
        return obs, rew, done, {"opp_action": opp_action, "raw_rew": raw}


# ── N-step buffer ─────────────────────────────────────────────────────────────
class NStepBuffer:
    def __init__(self, n: int = N_STEP, gamma: float = GAMMA):
        self.n = n; self.gamma = gamma
        self._s:  list = []; self._a: list = []; self._r: list = []
        self._s2: list = []; self._d: list = []
        self._ptr = 0   # instance-level — intentional, not class-level

    def push(self, s, a, r, s2, done):
        self._s.append(s); self._a.append(a); self._r.append(r)
        self._s2.append(s2); self._d.append(done)

    def _emit(self, start: int, end: int, replay: PERBuffer):
        R = 0.0
        for i in range(start, end):
            R += (self.gamma ** (i - start)) * self._r[i]
            if self._d[i]: break
        last = end - 1
        replay.push(self._s[start], self._a[start], R, self._s2[last], self._d[last])

    def flush(self, replay: PERBuffer):
        while len(self._s) - self._ptr >= self.n:
            self._emit(self._ptr, self._ptr + self.n, replay)
            self._ptr += 1

    def drain(self, replay: PERBuffer):
        while self._ptr < len(self._s):
            self._emit(self._ptr, len(self._s), replay)
            self._ptr += 1

    def clear(self):
        self._s.clear(); self._a.clear(); self._r.clear()
        self._s2.clear(); self._d.clear(); self._ptr = 0


# ── DQN Agent ─────────────────────────────────────────────────────────────────
class DQNAgent:
    name = "IPD-DQN-v2"

    def __init__(self):
        self.gamma      = GAMMA
        self.batch      = BATCH
        self.tgt_update = TGT_UPD
        self.steps      = 0
        self.beta_start = 0.4
        self.beta_end   = 1.0

        self._qnet_raw = DuelingDQN().to(device)
        self.tnet      = DuelingDQN().to(device)
        self.tnet.load_state_dict(self._qnet_raw.state_dict())
        self.tnet.eval()

        try:
            self.qnet = torch.compile(self._qnet_raw)
            _log("torch.compile enabled")
        except Exception:
            self.qnet = self._qnet_raw
            _log("torch.compile unavailable — using eager mode")

        self.opt   = optim.AdamW(self._qnet_raw.parameters(),
                                 lr=LR, weight_decay=1e-4, eps=1e-5)
        self.sched = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.opt, T_0=6_000, T_mult=2, eta_min=5e-6)

        self.buf   = PERBuffer()
        self.nstep = NStepBuffer()

    def beta(self) -> float:
        return self.beta_start + min(self.steps / BETA_STEPS, 1.0) * (self.beta_end - self.beta_start)

    def on_episode_start(self): self.nstep.clear()
    def on_episode_end(self):   self.nstep.drain(self.buf)

    def select_action(self, obs: np.ndarray, eval_mode: bool = False) -> int:
        with torch.no_grad():
            t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            if not eval_mode:
                self._qnet_raw.train(); self._qnet_raw.sample_noise()
            else:
                self._qnet_raw.eval()
            return int(self.qnet(t).argmax(1).item())

    def push(self, s, a, r, s2, done):
        self.nstep.push(s, a, r, s2, done)
        self.nstep.flush(self.buf)

    def learn(self) -> float:
        if len(self.buf) < self.batch: return 0.0

        b, idxs, weights = self.buf.sample(self.batch, beta=self.beta())
        S  = torch.tensor(np.array([t.s  for t in b]), dtype=torch.float32).to(device)
        A  = torch.tensor([t.a  for t in b], dtype=torch.long).to(device)
        R  = torch.tensor([t.r  for t in b], dtype=torch.float32).to(device)
        S2 = torch.tensor(np.array([t.s2 for t in b]), dtype=torch.float32).to(device)
        DN = torch.tensor([t.done for t in b], dtype=torch.float32).to(device)
        W  = torch.tensor(weights, dtype=torch.float32).to(device)

        self._qnet_raw.train(); self._qnet_raw.sample_noise()
        with torch.no_grad():
            na  = self.qnet(S2).argmax(1)                              # online selects action
            nq  = self.tnet(S2).gather(1, na.unsqueeze(1)).squeeze(1)  # target scores it
            tgt = R + (self.gamma ** N_STEP) * nq * (1 - DN)

        cur = self.qnet(S).gather(1, A.unsqueeze(1)).squeeze(1)
        td  = (cur - tgt).abs().detach().cpu().numpy()
        self.buf.update_prios(idxs, td)

        loss = (W * F.smooth_l1_loss(cur, tgt, reduction="none")).mean()
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self._qnet_raw.parameters(), 10.0)
        self.opt.step(); self.sched.step(); self.steps += 1

        if self.steps % self.tgt_update == 0:
            self.tnet.load_state_dict(self._qnet_raw.state_dict())

        return float(loss.item())


# ── Adaptive opponent sampler ──────────────────────────────────────────────────
class AdaptiveSampler:
    """Over-samples opponents where agent is weakest (hard-negative mining)."""
    def __init__(self, pool: list, alpha: float = 0.05):
        self.pool   = pool
        self.alpha  = alpha
        self.scores = {i: 200.0 for i in range(len(pool))}

    def sample(self):
        sc   = np.array([self.scores[i] for i in range(len(self.pool))])
        inv  = 1.0 / (sc + 1e-3)
        prob = inv / inv.sum()
        idx  = np.random.choice(len(self.pool), p=prob)
        return idx, self.pool[idx]

    def update(self, idx: int, ep_reward: float):
        self.scores[idx] = (1 - self.alpha) * self.scores[idx] + self.alpha * ep_reward


# ── Opponent pools ─────────────────────────────────────────────────────────────
def build_pools():
    # MetaWinner deliberately excluded: its Appeaser sub-player calls
    # opponent.history[-1] on round 0 causing IndexError in step-by-step env.
    train_specs = [
        (axl.Cooperator,           {},           "Cooperative"),
        (axl.TitForTat,            {},           "TFT-Like"),
        (axl.TitFor2Tats,          {},           "TFT-Like"),
        (axl.SuspiciousTitForTat,  {},           "TFT-Like"),
        (axl.HardTitForTat,        {},           "TFT-Like"),
        (axl.GTFT,                 {},           "TFT-Like"),
        (axl.SlowTitForTwoTats2,   {},           "TFT-Like"),
        (axl.OmegaTFT,             {},           "TFT-Like"),
        (axl.Grudger,              {},           "Grudger-Like"),
        (axl.SoftGrudger,          {},           "Grudger-Like"),
        (axl.Gradual,              {},           "Grudger-Like"),
        (axl.ForgetfulGrudger,     {},           "Grudger-Like"),
        (axl.EvolvedLookerUp1_1_1, {},           "Grudger-Like"),
        (axl.EvolvedLookerUp2_2_2, {},           "Grudger-Like"),
        (axl.EvolvedFSM16,         {},           "Evolved"),
        (axl.EvolvedHMM5,          {},           "Evolved"),
        (axl.Defector,             {},           "Defector-Like"),
        (axl.Prober,               {},           "Defector-Like"),
        (axl.HardProber,           {},           "Defector-Like"),
        (axl.Prober2,              {},           "Defector-Like"),
        (axl.TrickyDefector,       {},           "Defector-Like"),
        (axl.Aggravater,           {},           "Defector-Like"),
        (axl.Random,               {"p": 0.5},  "Stochastic"),
        (axl.Random,               {"p": 0.3},  "Stochastic"),
        (axl.GTFT,                 {"p": 0.33}, "Stochastic"),
        (axl.StochasticWSLS,       {"ep": 0.1}, "Stochastic"),
        (axl.StochasticWSLS,       {"ep": 0.2}, "Stochastic"),
        (axl.Alternator,           {},           "PhaseSwitching"),
        (axl.CyclerCCD,            {},           "PhaseSwitching"),
        (axl.CyclerDC,             {},           "PhaseSwitching"),
        (axl.CyclerCCCCCD,         {},           "PhaseSwitching"),
        (axl.AdaptiveTitForTat,    {},           "Adaptive"),
        (axl.EvolvedANN,           {},           "Evolved"),
    ]
    test_specs = [
        (axl.OmegaTFT,             {},           "TFT-Like"),
        (axl.ContriteTitForTat,    {},           "TFT-Like"),
        (axl.EvolvedLookerUp2_2_2, {},           "Grudger-Like"),
        (axl.EvolvedFSM16,         {},           "Evolved"),
        (axl.EvolvedHMM5,          {},           "Evolved"),
        (axl.StochasticWSLS,       {"ep": 0.2}, "Stochastic"),
        (axl.HardProber,           {},           "Defector-Like"),
        (axl.Prober2,              {},           "Defector-Like"),
        (axl.ForgetfulGrudger,     {},           "Grudger-Like"),
        (axl.AdaptiveTitForTat,    {},           "Adaptive"),
        (axl.Alternator,           {},           "PhaseSwitching"),
        (axl.CyclerCCCCCD,         {},           "PhaseSwitching"),
        (axl.EvolvedANN,           {},           "Evolved"),
    ]
    mk = lambda sp: [{"class": c, "kwargs": k, "family": f,
                      "name": str(c(**k))} for c, k, f in sp]
    return mk(train_specs), mk(test_specs)


# ── Evaluation ────────────────────────────────────────────────────────────────
STOCHASTIC_FAMILIES = {"Stochastic", "Evolved", "Adaptive"}

def _n_eval(e: dict) -> int:
    return 150 if e["family"] in STOCHASTIC_FAMILIES else 30

def evaluate(agent, pool: list, desc: str = "Eval", quiet: bool = False) -> dict:
    res      = {}
    iterable = tqdm(pool, desc=f"  {desc}", leave=False, ncols=90, unit="opp") \
               if not quiet else pool
    for e in iterable:
        n_eps = _n_eval(e)
        env   = IPDEnv(e["class"], e["kwargs"])
        rws   = []
        for _ in range(n_eps):
            agent.on_episode_start()
            obs = env.reset(); ep_r = 0.0
            for _ in range(ROUNDS):
                a = agent.select_action(obs, eval_mode=True)
                obs, rew, done, info = env.step(a)
                ep_r += info["raw_rew"]
                if done: break
            agent.on_episode_end()
            rws.append(ep_r)
        oname = f"{e['family']}/{e['name']}"
        res[oname] = {
            "avg":    float(np.mean(rws)),
            "std":    float(np.std(rws)),
            "median": float(np.median(rws)),
            "family": e["family"],
            "n_eps":  n_eps,
        }
    return res


# ── Snapshot ──────────────────────────────────────────────────────────────────
FAMILIES_ORDER = ["Cooperative", "TFT-Like", "Grudger-Like", "Evolved",
                  "Defector-Like", "Stochastic", "PhaseSwitching", "Adaptive"]

def _snapshot(ep: int, agent: DQNAgent, test_pool: list,
              rews_window: list, losses_window: list):
    res    = evaluate(agent, test_pool, desc=f"Snap @ep{ep}", quiet=False)
    mean_r = float(np.mean([v["avg"] for v in res.values()]))

    hdr  = f"\n{'─'*72}\n"
    hdr += f"  📊 SNAPSHOT  ep={ep:>6}  |  AvgRew(test)={mean_r:>6.1f}"
    hdr += f"  |  Loss={np.mean(losses_window) if losses_window else 0:.4f}"
    hdr += f"  |  β={agent.beta():.3f}  |  LR={agent.opt.param_groups[0]['lr']:.2e}"
    tqdm.write(hdr)

    fam_scores: Dict[str, list] = {}
    for v in res.values():
        fam_scores.setdefault(v["family"], []).append(v["avg"])
    row = "  "
    for fam in FAMILIES_ORDER:
        if fam in fam_scores:
            sc    = np.mean(fam_scores[fam])
            short = fam.replace("-Like", "").replace("Phase", "Ph")[:8]
            row  += f"  {short}:{sc:>5.1f}"
    tqdm.write(row)
    tqdm.write(f"{'─'*72}")
    return res, mean_r


# ── Baselines ─────────────────────────────────────────────────────────────────
def policy_tft(ph, oh, r):    return C if not oh else oh[-1]
def policy_grim(ph, oh, r):   return D if any(a == D for a in oh) else C
def policy_gtft(ph, oh, r):
    if not oh: return C
    return (D if random.random() > 0.15 else C) if oh[-1] == D else C
def policy_tft2(ph, oh, r):   return D if len(oh) >= 2 and oh[-1] == D and oh[-2] == D else C
def policy_pavlov(ph, oh, r):
    if not ph: return C
    return C if (ph[-1] == C and oh[-1] == C) or (ph[-1] == D and oh[-1] == D) else D

class _HC:
    def __init__(self, fn, name):
        self._fn = fn; self.name = name
        self._ph: list = []; self._oh: list = []
    def on_episode_start(self): self._ph = []; self._oh = []
    def on_episode_end(self): pass
    def select_action(self, obs, eval_mode=False):
        return self._fn(self._ph, self._oh, len(self._ph))

def make_baselines():
    return [
        _HC(policy_tft,        "TFT"),
        _HC(policy_grim,       "Grim"),
        _HC(policy_gtft,       "GTFT"),
        _HC(policy_tft2,       "TFT2"),
        _HC(policy_pavlov,     "Pavlov"),
        _HC(lambda p,o,r: C,   "AllC"),
        _HC(lambda p,o,r: D,   "AllD"),
    ]


# ── Final results table ────────────────────────────────────────────────────────
def print_final_table(all_res: dict, pool: list) -> dict:
    names  = list(all_res.keys())
    onames = list(next(iter(all_res.values())).keys())
    W      = 9
    sep    = "═" * max(36 + W * len(names), 80)
    hdr    = f"{'Opponent':<36}" + "".join(f"{n:>{W}}" for n in names)

    tqdm.write(f"\n{sep}")
    tqdm.write("  FINAL RESULTS  —  avg raw reward per episode (100 rounds)")
    tqdm.write(f"  Mutual coop = 300 pts  |  Exploit coop = 500 pts  |  Mutual defect = 100 pts")
    tqdm.write(f"{sep}")
    tqdm.write(hdr); tqdm.write("─" * len(hdr))

    fam_scores = {n: {} for n in names}
    for o in onames:
        sh   = o.split("/")[-1][:34]
        vs   = {n: all_res[n][o]["avg"] for n in names}
        best = max(vs.values())
        row  = f"{sh:<36}"
        for n in names:
            mk = "*" if abs(vs[n] - best) < 0.5 else " "
            row += f"{vs[n]:>{W-1}.1f}{mk}"
            fam_scores[n].setdefault(all_res[n][o]["family"], []).append(vs[n])
        tqdm.write(row)

    tqdm.write("─" * len(hdr))
    row = f"{'OVERALL MEAN':<36}"
    for n in names:
        row += f"{np.mean([all_res[n][o]['avg'] for o in onames]):>{W}.1f}"
    tqdm.write(row)

    all_fams = sorted({e["family"] for e in pool})
    for fam in all_fams:
        row = f"  {fam:<34}"
        for n in names:
            row += f"{np.mean(fam_scores[n].get(fam, [0])):>{W}.1f}"
        tqdm.write(row)

    tqdm.write(sep)
    tqdm.write("* = within 0.5 of best for that opponent\n")

    means = {n: float(np.mean([all_res[n][o]["avg"] for o in onames])) for n in names}
    dqn_m = means.get("IPD-DQN-v2", 0)
    tqdm.write("  📈 Ranking by overall mean reward:")
    for n, m in sorted(means.items(), key=lambda x: -x[1]):
        marker = "  ◀ OUR AGENT" if n == "IPD-DQN-v2" else ""
        delta  = f"  (DQN {dqn_m - m:+.1f})" if n != "IPD-DQN-v2" else ""
        tqdm.write(f"    {n:<14}: {m:>6.1f}{delta}{marker}")
    tqdm.write(f"\n{sep}\n")
    return means


# ── Training ──────────────────────────────────────────────────────────────────
def train_agent(agent: DQNAgent, pool: list, test_pool: list) -> dict:
    _log(f"Training {TOTAL_EPS} episodes | warmup={WARMUP_EPS} | device={device}")
    t0 = time.time()

    warm_pool = [e for e in pool if e["family"] in ("Cooperative", "TFT-Like")]
    sampler   = AdaptiveSampler(pool)
    rews:     list = []
    losses:   list = []
    snapshots: dict = {}

    outer = tqdm(
        range(TOTAL_EPS), desc="Training", ncols=100, unit="ep",
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
    )

    for ep in outer:
        if ep < WARMUP_EPS:
            e = random.choice(warm_pool); idx = None
        else:
            idx, e = sampler.sample()

        env = IPDEnv(e["class"], e["kwargs"])
        obs = env.reset()
        agent.on_episode_start(); ep_raw = 0.0

        for _ in range(ROUNDS):
            a = agent.select_action(obs)
            obs2, rew, done, info = env.step(a)
            agent.push(obs, a, rew, obs2, done)
            l = agent.learn()
            if l: losses.append(l)
            obs = obs2; ep_raw += info["raw_rew"]
            if done: break

        agent.on_episode_end()
        rews.append(ep_raw)
        if idx is not None: sampler.update(idx, ep_raw)

        # update postfix every 200 eps (cheap)
        if (ep + 1) % 200 == 0:
            outer.set_postfix({
                "AvgR": f"{np.mean(rews[-500:]):.1f}",
                "Loss": f"{np.mean(losses[-200:]) if losses else 0:.4f}",
                "β":    f"{agent.beta():.3f}",
                "LR":   f"{agent.opt.param_groups[0]['lr']:.2e}",
                "buf":  len(agent.buf),
            }, refresh=False)

        # verbose snapshot
        if (ep + 1) % VERBOSE_EVERY == 0:
            snap_res, snap_mean = _snapshot(
                ep + 1, agent, test_pool,
                rews[-VERBOSE_EVERY:], losses[-2000:]
            )
            snapshots[ep + 1] = snap_mean

    _log(f"Training done in {(time.time()-t0)/60:.1f}m")
    return snapshots


# ── main ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    t0 = time.time()
    os.makedirs("./results", exist_ok=True)

    print(f"\n{'═'*72}")
    print(f"  IPD-DQN v2  |  Publishable Quality  |  Device: {device}")
    print(f"  Episodes: {TOTAL_EPS}  |  Warmup: {WARMUP_EPS}  |  Hidden: {HIDDEN}  |  FEAT: {FEAT_DIM}")
    print(f"  Batch: {BATCH}  |  Buffer: {BUF_CAP}  |  TgtUpd: {TGT_UPD}  |  LR: {LR}")
    print(f"{'═'*72}\n")

    train_pool, test_pool = build_pools()
    _log(f"Train pool: {len(train_pool)} opponents  |  Test pool: {len(test_pool)} opponents")
    _log("MetaWinner excluded (sub-player IndexError in step-by-step env)")

    agent     = DQNAgent()
    snapshots = train_agent(agent, train_pool, test_pool)

    print()
    _log("Running final evaluation...")
    all_res = {"IPD-DQN-v2": evaluate(agent, test_pool, desc="DQN Final")}
    for bl in tqdm(make_baselines(), desc="  Baselines", ncols=80, unit="agent"):
        all_res[bl.name] = evaluate(bl, test_pool, quiet=True)

    means = print_final_table(all_res, test_pool)

    out = {
        "means":     means,
        "snapshots": snapshots,
        "results": {
            ag: {o: {"avg": v["avg"], "std": v["std"],
                     "median": v["median"], "family": v["family"]}
                 for o, v in r.items()}
            for ag, r in all_res.items()
        },
        "config": {
            "feat_dim":        FEAT_DIM,
            "n_step":          N_STEP,
            "gamma":           GAMMA,
            "hidden":          HIDDEN,
            "train_episodes":  TOTAL_EPS,
            "warmup":          WARMUP_EPS,
            "batch":           BATCH,
            "tgt_update":      TGT_UPD,
            "buf_cap":         BUF_CAP,
            "lr":              LR,
            "fixes": [
                "seed_player_uses_axelrod_RandomGenerator_not_numpy_RandomState",
                "MetaWinner_excluded_incompatible_step_env",
                "NStepBuffer_ptr_instance_level",
            ],
            "improvements": [
                "reward_shaping_normalised_plus_coop_bonus",
                "deeper_network_256_hidden_3_layer_GELU",
                "AdamW_weight_decay_1e-4",
                "PER_alpha_0.7",
                "beta_annealed_60k_steps",
                "150_eval_eps_stochastic_opponents",
                "cosine_LR_warm_restarts_T0_6000",
                "tqdm_progress_bars_with_postfix",
                "snapshots_every_2000_eps",
            ],
        },
    }
    with open("./results/ipd_dqn_v2.json", "w") as f:
        json.dump(out, f, indent=2)

    _log(f"Total wall time: {(time.time()-t0)/60:.1f}m")
    _log("Saved → ./results/ipd_dqn_v2.json")


════════════════════════════════════════════════════════════════════════
  IPD-DQN v2  |  Publishable Quality  |  Device: cuda
  Episodes: 20000  |  Warmup: 4000  |  Hidden: 256  |  FEAT: 32
  Batch: 512  |  Buffer: 150000  |  TgtUpd: 1000  |  LR: 0.0003
════════════════════════════════════════════════════════════════════════

  [IPD] Train pool: 33 opponents  |  Test pool: 13 opponents
  [IPD] MetaWinner excluded (sub-player IndexError in step-by-step env)
  [IPD] torch.compile enabled
  [IPD] Training 20000 episodes | warmup=4000 | device=cuda


Training:  10%|████▍                                       | 2000/20000 [41:18<126:28:17, 25.29s/ep]


────────────────────────────────────────────────────────────────────────
  📊 SNAPSHOT  ep=  2000  |  AvgRew(test)= 235.3  |  Loss=0.0044  |  β=1.000  |  LR=2.96e-04
    TFT:299.0  Grudger:245.0  Evolved:268.0  Defector:152.5  Stochast:153.3  PhSwitch:202.5  Adaptive:303.0
────────────────────────────────────────────────────────────────────────


Training:  20%|████████▍                                 | 4000/20000 [1:27:07<108:43:37, 24.46s/ep]


────────────────────────────────────────────────────────────────────────
  📊 SNAPSHOT  ep=  4000  |  AvgRew(test)= 236.4  |  Loss=0.0034  |  β=1.000  |  LR=2.98e-04
    TFT:301.0  Grudger:245.0  Evolved:267.1  Defector:152.5  Stochast:163.3  PhSwitch:204.5  Adaptive:303.0
────────────────────────────────────────────────────────────────────────


...


────────────────────────────────────────────────────────────────────────
  📊 SNAPSHOT  ep= 18000  |  AvgRew(test)= 309.8  |  Loss=0.0051  |  β=1.000  |  LR=2.78e-04
    TFT:302.0  Grudger:302.0  Evolved:302.0  Defector:300.0  Stochast:288.9  PhSwitch:361.0  Adaptive:302.0
────────────────────────────────────────────────────────────────────────


Training: 100%|█████████████████████████████████████████████| 20000/20000 [7:26:51<00:00,  1.34s/ep]



────────────────────────────────────────────────────────────────────────
  📊 SNAPSHOT  ep= 20000  |  AvgRew(test)= 293.3  |  Loss=0.0043  |  β=1.000  |  LR=2.37e-04
    TFT:302.0  Grudger:302.0  Evolved:302.0  Defector:193.5  Stochast:287.5  PhSwitch:361.0  Adaptive:302.0
────────────────────────────────────────────────────────────────────────
  [IPD] Training done in 446.9m

  [IPD] Running final evaluation...


  Baselines: 100%|█████████████████████████████| 7/7 [00:20<00:00,  2.97s/agent]


════════════════════════════════════════════════════════════════════════════════════════════════════════════
  FINAL RESULTS  —  avg raw reward per episode (100 rounds)
  Mutual coop = 300 pts  |  Exploit coop = 500 pts  |  Mutual defect = 100 pts
════════════════════════════════════════════════════════════════════════════════════════════════════════════
Opponent                            IPD-DQN-v2      TFT     Grim     GTFT     TFT2   Pavlov     AllC     AllD
─────────────────────────────────────────────────────────────────────────────────────────────────────────────
Omega TFT: 3, 8                        302.0*   300.0    300.0    300.0    300.0    300.0    300.0    104.0 
Contrite Tit For Tat                   302.0*   300.0    300.0    300.0    300.0    300.0    300.0    104.0 
EvolvedLookerUp2_2_2                   302.0*   300.0    300.0    300.0    300.0    300.0    300.0    108.0 
Evolved FSM 16                         302.0    300.0    300.0    300.0    300.0    300.0    30